In [1]:
import numpy as np
from langchain_core.runnables import RunnableLambda
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
from openai import OpenAI
import httpx
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field          # Field from pydantic, NOT dataclasses
from typing import Literal, TypedDict, List
import uuid
import base64
import ssl
import json
from langgraph.graph import StateGraph, START, END
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.embeddings import Embeddings
from langchain_core.documents import Document

load_dotenv()
ssl_cert_file = "C:\\Users\\Ujjwal_Manna\\Downloads\\ca-certs.crt"
os.environ['REQUESTS_CA_BUNDLE'] = ssl_cert_file
os.environ['SSL_CERT_FILE'] = ssl_cert_file

client        = httpx.Client(verify=ssl.create_default_context(cafile=ssl_cert_file))
client_id     = os.environ.get('LLM_CLIENT_ID')
client_secret = os.environ.get('LLM_CLIENT_SECRET')
auth_header   = f"Basic {base64.b64encode(f'{client_id}:{client_secret}'.encode()).decode()}"

llm_openai = ChatOpenAI(
    model="gpt-oss-120b",
    base_url=os.environ.get("LLM_URL"),
    http_client=client,
    api_key="dummy value",
    default_headers={
        'accept': '*/*',
        'Content-Type': 'application/json',
        'Authorization': auth_header
    },
    temperature=0.0,
    streaming=True
)

# ── Embedding model ────────────────────────────────────────────────────────────
class DSXEmbeddingInterface(Embeddings):
    def __init__(self, url, model_name, http_client, default_headers, batch_size=32):
        self.client = OpenAI(
            base_url=url, http_client=http_client,
            api_key='not-empty', default_headers=default_headers
        )
        self.model_name = model_name
        self.batch_size = batch_size

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        all_embeddings = []
        for i in range(0, len(texts), self.batch_size):
            batch = texts[i:i + self.batch_size]
            resp  = self.client.embeddings.create(model=self.model_name, input=batch)
            all_embeddings.extend([a.embedding for a in resp.data])
        return all_embeddings

    def embed_query(self, text: str) -> List[float]:
        return self.embed_documents([text])[0]

embedding_model = DSXEmbeddingInterface(
    url="",
    model_name="",
    http_client=httpx.Client(verify=ssl.create_default_context(cafile=ssl_cert_file)),
    default_headers={
        "x-correlation-id": str(uuid.uuid4()),
        'accept': '*/*', 'Content-Type': 'application/json',
        'Authorization': auth_header
    },
    batch_size=32
)

# ── Vector store ───────────────────────────────────────────────────────────────
class SimpleVectorStore:
    def __init__(self, documents=None, embedding_model=None, persist_directory=None):
        self.documents        = documents if documents is not None else []
        self.embedding_model  = embedding_model
        self.embeddings       = None
        self.persist_directory = persist_directory

    def add_documents(self):
        texts    = [doc.page_content for doc in self.documents]
        new_embs = np.array(self.embedding_model.embed_documents(texts))
        self.embeddings = new_embs if self.embeddings is None else np.vstack([self.embeddings, new_embs])

    def add_documents_to_existing(self, new_documents):
        texts    = [doc.page_content for doc in new_documents]
        new_embs = np.array(self.embedding_model.embed_documents(texts))
        self.documents.extend(new_documents)
        self.embeddings = new_embs if self.embeddings is None else np.vstack([self.embeddings, new_embs])

    def similarity_search(self, query, k=3):
        q_emb   = np.array(self.embedding_model.embed_query(query))
        q_norm  = np.linalg.norm(q_emb)
        d_norms = np.linalg.norm(self.embeddings, axis=1)
        sims    = np.dot(self.embeddings, q_emb) / (d_norms * q_norm)
        return [self.documents[i] for i in np.argsort(sims)[::-1][:k]]

    def save(self, directory=None):
        save_dir = directory or self.persist_directory
        if not save_dir:
            raise ValueError("No persist directory specified")
        os.makedirs(save_dir, exist_ok=True)
        np.save(os.path.join(save_dir, "embeddings.npy"), self.embeddings)
        docs_data = [{"page_content": d.page_content, "metadata": d.metadata} for d in self.documents]
        with open(os.path.join(save_dir, "documents.json"), "w") as f:
            json.dump(docs_data, f)
        print(f"Vector store saved to {save_dir}")

    @classmethod
    def load(cls, directory, embedding_model):
        embeddings = np.load(os.path.join(directory, "embeddings.npy"))
        with open(os.path.join(directory, "documents.json")) as f:
            docs_data = json.load(f)
        documents = [Document(page_content=d["page_content"], metadata=d["metadata"]) for d in docs_data]
        vs = cls(documents=documents, embedding_model=embedding_model, persist_directory=directory)
        vs.embeddings = embeddings
        print(f"Vector store loaded from {directory} ({len(documents)} docs)")
        return vs

# ── Load vector store ──────────────────────────────────────────────────────────
persist_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "vector_store"))
if os.path.exists(persist_dir) and os.path.exists(os.path.join(persist_dir, "embeddings.npy")):
    vectorstore = SimpleVectorStore.load(persist_dir, embedding_model)
else:
    loader      = PyPDFLoader("../../Docs/fabric-admin.pdf")
    new_docs    = loader.load()
    vectorstore = SimpleVectorStore(new_docs, embedding_model, persist_directory=persist_dir)
    vectorstore.add_documents()
    vectorstore.save()

# ══════════════════════════════════════════════════════════════════════════════
# YOUR CODE — written from scratch
# ══════════════════════════════════════════════════════════════════════════════

def cosine_similarity(A, B):
    dot_p  = np.array(A).dot(np.array(B))
    mag_a  = np.sqrt(np.sum(np.square(A)))
    mag_b  = np.sqrt(np.sum(np.square(B)))
    return dot_p / (mag_a * mag_b)          # parentheses around denominator — critical


def similarity_search(query_text, documents, embeddings, embedding_model, k=3):
    q_emb  = np.array(embedding_model.embed_query(query_text))
    scores = []
    for i in range(len(documents)):
        score = cosine_similarity(q_emb, embeddings[i])
        scores.append(score)
    top_k_indices = np.argsort(scores)[::-1][:k]
    # FIX: return Document objects, NOT the index numbers
    # result was numpy.int64 indices — you need documents[i], not i itself
    return [documents[i] for i in top_k_indices]

def similarity_search1(query_text, documents, embeddings, embedding_model, k=3):
    q_emb  = np.array(embedding_model.embed_query(query_text))
    scores = []
    d_norms=np.linalg.norm(embeddings,axis=1)
    q_norm=np.linalg.norm(q_emb)
    sims=np.dot(embeddings,q_emb)/(d_norms*q_norm)
    top_k_indices = np.argsort(sims)[::-1][:k]
    # FIX: return Document objects, NOT the index numbers
    # result was numpy.int64 indices — you need documents[i], not i itself
    return [documents[i] for i in top_k_indices]


# ── Verify: your output must match vectorstore.similarity_search ──────────────
my_results    = similarity_search1(
    "What is OneLake?",
    vectorstore.documents,
    vectorstore.embeddings,
    embedding_model,
    k=3
)
their_results = vectorstore.similarity_search("What is OneLake?", k=3)

print("=== Comparison (pages must match) ===")
for i in range(3):
    mine   = my_results[i].metadata.get('page', '?')
    theirs = their_results[i].metadata.get('page', '?')
    match  = "✅" if mine == theirs else "❌"
    print(f"[{i+1}] {match}  MINE={mine}  THEIRS={theirs}")


C:\Users\Ujjwal_Manna\AppData\Local\Temp\ipykernel_46900\2336090055.py:17: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Vector store loaded from C:\projects\personal\langchn_practise\vector_store (1160 docs)
=== Comparison (pages must match) ===
[1] ✅  MINE=289  THEIRS=289
[2] ✅  MINE=289  THEIRS=289
[3] ✅  MINE=289  THEIRS=289
